# SupeRR Selector — player ratings

This notebook reads the silver files and builds two lists: batters and bowlers.

Each player has a baseline from their usual rates, then three pressure bands, a within-match comparison, form, and consistency. Clutch is shown beside the rating and is not part of the total, because it repeats the high band. A missed season is left out of a rate. Each ball keeps its season weight: 0.6561, 0.729, 0.81, 0.9, and 1 for 2022 through 2026. The seven composite weights keep those ratios and sum to 1.

The calculations live in `player_ratings.py`. This notebook runs them and shows the tables. The silver files are not changed.

## Load and tag balls

A legal ball is not a wide, a no-ball, or a super over. A wide or a no-ball adds one conceded run. Bye and leg-bye amounts are not in the file, and neither are extra runs beyond that one-run penalty, so a team total is still a lower bound. An ordinary chase target is the first-innings total plus one. A DLS target is used as supplied.

Situation points come from the phase, wickets down, a knockout or a final, rain, and one chase term worth 0, 1, or 2 from the required rate. The powerplay is worth 2 and the death is worth 3. Two wickets down are worth 2, and further wickets add nothing, so the tail is not treated as a harder ball than an early collapse. A death over with two wickets down is high while setting a total. The same total is the same band in both innings: zero is 0–2, medium is 3–4, and high is 5 or more.

In [1]:
from pathlib import Path

import pandas as pd

import player_ratings as ratings

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 30)

balls, matches, comps = ratings.load()
work, legal = ratings.add_situation(balls)
print("deliveries", len(balls), "legal, excluding super overs", len(legal))
print("component weights sum to", round(sum(ratings.COMPONENT_WEIGHTS.values()), 6))
bands = ratings.band_summary(legal)
display(bands.assign(share=lambda frame: (100 * frame["share"]).round(1)))
mix = (
    legal.groupby(["innings", "band"]).size().unstack(fill_value=0).reindex(columns=ratings.BANDS)
)
display(mix)

deliveries 200999 legal, excluding super overs 192675
component weights sum to 1.0


,balls,share
band,,
zero,99703,51.7
medium,53947,28.0
high,39025,20.3


band,zero,medium,high
innings,,,
1,72029,9577,19723
2,27674,44370,19302


## Elo

Expected runs and wicket chances are fit by competition, phase, pace or spin, and how many wickets are already down. A thin cell falls back to the same situation without the wicket count, then to the IPL or international ladder, then to the phase. The batter uses the chance the striker is dismissed. The bowler uses the chance of a bowler wicket on a legal ball, and the much smaller chance on a wide or no-ball. Team strength updates after each match. Batting strength updates after each legal ball, and bowling strength also updates for a wide, a no-ball, and a wicket on an illegal ball. Player ratings stay on separate IPL and international ladders. Venue strength uses only balls already seen at that ground. A ball is worth the runs and the wicket chance minus what those ratings expected.

In [2]:
mu, p_out, p_bowl, rho = ratings.expectation_tables(legal)
work = ratings.fit_elo(work, mu, p_out, p_bowl)
legal = work.loc[work["is_legal"]].copy()
summary = work.attrs["elo_summary"]
for key, value in summary.items():
    if isinstance(value, str):
        print(key + ":", value)
    else:
        print(key + ":", round(value, 1))
print("mean bat runs per legal ball", round(rho, 3))
print("batter residual", legal.groupby("ladder")["e_bat"].mean().round(4).to_dict())
print("bowler residual", work.groupby("ladder")["e_bowl"].mean().round(4).to_dict())

k_run: 4.0
k_wicket: 16.0
k_team: 20.0
batter_elo_min: 1051.2
batter_elo_max: 2491.4
batter_elo_sd: 129.4
bowler_elo_min: 1124.1
bowler_elo_max: 2098.6
bowler_elo_sd: 112.9
team_elo_min: 1361.3
team_elo_max: 1647.0
grounds: 97
step_sizes: as written
mean bat runs per legal ball 1.394
batter residual {'INT': -0.0009, 'IPL': -0.0009}
bowler residual {'INT': 0.0007, 'IPL': 0.0012}


## Batter scores

Baseline is strike rate and average against the competition and the phases that batter actually plays. A thin competition cell falls back to the ladder, then the phase. The three bands are the strength-adjusted ball value in that band minus the batter's own overall value, so a player who is equally good everywhere is not credited again. Peers compare an innings with teammates and the opposition, and both comparisons stop at 3. Clutch is a high-band passage against the batter's own strike rate: better than that rate is positive. Form is the last eight qualifying innings against the strike rate from earlier balls only. Consistency prefers a narrower spread of match values.

Baseline, peers, form, and consistency are z-scores on players with at least 120 decay-weighted legal balls. Each band, and clutch, is a z-score on players with at least 30 decay-weighted balls in that band. Each z-score stops at 3. The composite uses the baseline, zero, medium, high, peers, form, and consistency weights in the ratio 0.15, 0.10, 0.15, 0.20, 0.10, 0.05, and 0.05. Clutch stays in the table and is left out of that sum.

In [3]:
batters = ratings.score_batters(legal)
print("batters", len(batters), "ranked", int(batters["ranked"].sum()))
batter_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "runs", "strike_rate", "average", "n_eff", "high_n_eff",
]
display(ratings.top_table(batters, batter_cols))
print("roles in the batter top 10")
print(batters.loc[batters["rank"].le(10), "role"].value_counts().to_string())

batters 888 ranked 196


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,runs,strike_rate,average,n_eff,high_n_eff
0,1.0,Maaz Sadaqat,top order,0.728,2.507,0.767,-0.681,-0.541,-0.547,2.864,-0.180,1.240,404,177.193,80.800,213.500,34.100
1,2.0,Tim David,middle order,0.659,1.864,-1.652,-0.058,0.945,0.991,2.664,-0.328,-0.345,2153,177.202,34.726,1002.709,415.701
2,3.0,Virat Kohli,top order,0.652,1.107,-0.477,0.776,1.111,0.951,-0.670,1.147,1.481,3566,145.255,47.547,1970.729,132.636
3,4.0,Fakhar Zaman,top order,0.648,-0.010,-2.427,1.577,2.505,1.865,0.167,-0.014,0.189,1127,131.505,26.209,724.595,92.159
4,5.0,Suryakumar Yadav,middle order,0.635,1.908,-0.360,0.290,0.453,-0.279,1.700,-0.990,0.064,4578,166.473,39.128,2167.299,341.241
5,6.0,Yashasvi Jaiswal,top order,0.631,0.621,-0.843,1.384,1.087,1.666,0.755,-0.377,0.290,2905,155.764,35.864,1517.598,48.466
6,7.0,Jos Buttler,top order,0.629,1.242,-0.022,-0.144,1.182,2.684,0.639,0.593,0.223,4384,155.461,41.752,2303.778,178.070
7,8.0,Shashank Singh,middle order,0.587,1.343,-3.000,0.924,1.629,1.298,1.196,1.407,-1.721,905,161.607,37.708,480.187,216.910
8,9.0,Sediqullah Atal,top order,0.457,0.194,-0.257,-1.097,2.825,2.389,-0.153,-1.130,0.670,1050,128.049,30.882,717.731,46.730
9,10.0,Sanju Samson,top order,0.457,0.579,-0.160,0.178,1.058,0.617,0.191,0.690,0.057,3436,154.984,32.415,1855.308,131.584


roles in the batter top 10
role
top order       7
middle order    3


## Bowler scores

The same shape. Baseline uses economy and balls per wicket, against the usual rate for that bowler's competition, pace or spin, and the phases they bowl. A thin cell falls back to the ladder, then the phase. Economy counts one run for each wide and each no-ball. A lower rate is a higher score. Each band is the bowler's value in that band minus their own overall value. Clutch compares a high-band spell with the bowler's own economy and is not part of the total.

In [4]:
bowlers = ratings.score_bowlers(work)
print("bowlers", len(bowlers), "ranked", int(bowlers["ranked"].sum()))
bowler_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "wickets", "economy", "balls_per_wicket", "n_eff", "high_n_eff",
]
display(ratings.top_table(bowlers, bowler_cols))
print("roles in the bowler top 10")
print(bowlers.loc[bowlers["rank"].le(10), "role"].value_counts().to_string())

bowlers 647 ranked 211


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,wickets,economy,balls_per_wicket,n_eff,high_n_eff
0,1.0,Sunil Narine,Spin,1.039,1.064,-0.363,-0.390,1.761,0.681,3.000,0.878,1.407,64.0,6.863,23.906,1246.702,91.612
1,2.0,Jasprit Bumrah,Pace,0.723,1.384,-0.278,-0.816,1.033,0.850,2.878,0.574,-0.046,85.0,7.169,20.106,1455.083,472.101
2,3.0,Matthew Humphreys,Spin,0.650,0.874,-0.157,-0.862,1.562,1.973,1.544,0.385,0.956,20.0,7.062,15.250,277.028,37.300
3,4.0,Ahmed Daniyal,Pace,0.630,1.458,0.135,0.452,-0.341,0.930,1.905,0.577,1.060,10.0,5.228,17.100,152.010,63.090
4,5.0,Shakib Al Hasan,Spin,0.564,1.013,-0.042,-0.527,0.557,0.004,1.631,1.273,0.885,20.0,6.706,17.000,241.088,47.029
5,6.0,Sufiyan Muqeem,Spin,0.546,1.552,-1.462,1.482,-0.257,-0.425,1.838,1.373,-1.462,39.0,6.000,13.282,449.880,52.020
6,7.0,Abrar Ahmed,Spin,0.538,0.948,-0.884,0.772,0.160,0.978,1.752,0.966,0.101,46.0,6.660,16.609,689.250,117.270
7,8.0,Mohammad Nawaz,Spin,0.515,0.986,-0.734,0.352,0.800,0.931,0.953,0.968,-0.385,66.0,7.124,15.212,812.210,56.584
8,9.0,Nathan Ellis,Pace,0.488,0.922,0.008,0.407,-0.164,0.241,1.020,1.345,1.086,58.0,8.231,15.810,736.037,288.675
9,10.0,Akeal Hosein,Spin,0.448,0.072,-0.395,-0.019,1.207,0.887,1.797,-0.396,-0.225,79.0,7.708,21.430,1389.741,85.551


roles in the bowler top 10
role
Spin    7
Pace    3


## Rankings

A player is ranked in a skill with at least 120 decay-weighted legal balls, and at least 30 of those in the high band. High now includes the powerplay after early wickets as well as the death, so a top-order career can clear that line. Anyone under the line stays in the full table with no rank. The checks confirm the band cuts, that the high band is not reserved for the tail, the residuals, and the concession. The two lists are saved under `ratings/`.

In [5]:
checks = ratings.sanity_report(work, legal)
for key, value in checks.items():
    print(key + ":", value)
ranked_batters = batters.loc[batters["ranked"]]
medium_link = float(ranked_batters["baseline"].corr(ranked_batters["band_medium"]))
print("ranked batter baseline-medium correlation", round(medium_link, 3))
if abs(medium_link) >= 0.5:
    raise AssertionError("the medium band still tracks the baseline")
out = Path("ratings")
out.mkdir(exist_ok=True)
batters.to_csv(out / "batters.csv", index=False)
bowlers.to_csv(out / "bowlers.csv", index=False)
print("wrote", out / "batters.csv", "and", out / "bowlers.csv")
print("decision: floor is 120 decay-weighted legal balls and 30 high-band balls")
print("decision: elo step sizes", summary["step_sizes"])

career_rate_of_ones: 1.0
high_chase_share: 0.4946060217809097
high_first_innings: 19723
league_first_innings_high: 18258
high_top_order_share: 0.1419859064702114
high_tail_share: 0.4051761691223575
wide_residual: -0.0010319513250684133
weight_sum: 0.9999999999999999
batter_residual: {'INT': -0.0008935894070006363, 'IPL': -0.0009369611688800129}
bowler_residual: {'INT': 0.0006645457151390909, 'IPL': 0.0012158335585030725}
conceded: 277791.0
legal_batter_runs: 268654.0
ranked batter baseline-medium correlation -0.044
wrote ratings\batters.csv and ratings\bowlers.csv
decision: floor is 120 decay-weighted legal balls and 30 high-band balls
decision: elo step sizes as written
